In [ ]:
# -*- coding: utf-8 -*-
"""
Enhanced Marathi Plagiarism Detection System - Fixed Version
Author: AI Assistant
Version: 2.2.0
"""

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling
import re
import string
import warnings
from typing import List, Tuple, Dict, Any, Optional, Union
import json
import logging
from dataclasses import dataclass
from pathlib import Path
import joblib
from functools import lru_cache
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, as_completed
import unicodedata
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

warnings.filterwarnings('ignore')

@dataclass
class PlagiarismConfig:
    """Configuration for plagiarism detection"""
    tfidf_weight: float = 0.4
    bert_weight: float = 0.6
    threshold: float = 0.55
    max_features: int = 8000
    min_df: int = 2
    max_df: float = 0.95
    ngram_range: Tuple[int, int] = (1, 3)
    batch_size: int = 32
    use_faiss: bool = False
    cache_embeddings: bool = True
    enable_parallel_processing: bool = True

class MarathiTextProcessor:
    """Enhanced Marathi text preprocessing"""

    def __init__(self):
        # Extended Marathi stopwords with frequency-based selection
        self.marathi_stopwords = self._load_comprehensive_stopwords()

        # Marathi-specific punctuation and characters
        self.marathi_punctuation = string.punctuation + '।॥॰'

        # Unicode ranges for Devanagari script
        self.devanagari_range = r'[\u0900-\u097F]'

        # Common Marathi suffixes and prefixes for stemming
        self.marathi_suffixes = ['ला', 'ने', 'चा', 'ची', 'चे', 'मधे', 'वर', 'खाली']
        self.marathi_prefixes = ['अ', 'आ', 'वि', 'दु', 'नि', 'सु']

    def _load_comprehensive_stopwords(self) -> set:
        """Load comprehensive Marathi stopwords"""
        base_stopwords = {
            'आणि', 'ते', 'ती', 'तो', 'त्या', 'तुमच्या', 'तुम्ही', 'मी', 'आहे', 'आहेत',
            'की', 'पण', 'म्हणून', 'होता', 'होती', 'होते', 'असून', 'किंवा', 'परंतु',
            'तर', 'मग', 'जर', 'तरी', 'काय', 'कोण', 'कसे', 'केव्हा', 'का', 'कुठे',
            'कोणते', 'फक्त', 'सगळे', 'आज', 'उद्या', 'काल', 'येथे', 'तेथे', 'सर्व',
            'अनेक', 'दोन', 'तीन', 'चार', 'पाच', 'दहा', 'शेकडो', 'हजारो', 'लाख',
            'अशी', 'असे', 'आपण', 'आपल्या', 'त्यानी', 'त्यांनी', 'तुझी', 'माझा',
            'माझी', 'माझे', 'तुझा', 'तुझे', 'त्याचा', 'त्याची', 'त्याचे', 'आमचा',
            'आमची', 'आमचे', 'त्यांचा', 'त्यांची', 'त्यांचे', 'हा', 'ही', 'हे', 'या'
        }

        # Add common function words and particles
        extended_stopwords = {
            'अधिक', 'कमी', 'समान', 'वेगळे', 'सारखे', 'प्रत्येक', 'काही', 'सगळे',
            'सर्व', 'एक', 'दोन', 'तीन', 'चार', 'पाच', 'सहा', 'सात', 'आठ', 'नऊ', 'दहा',
            'माझ्या', 'तुझ्या', 'त्याच्या', 'तिच्या', 'त्यांच्या', 'आमच्या', 'तुमच्या',
            'या', 'ता', 'ना', 'स', 'ल', 'च', 'शी', 'साठी', 'बद्दल', 'विषयी'
        }

        return base_stopwords.union(extended_stopwords)

    def normalize_text(self, text: str) -> str:
        """Normalize Marathi text with advanced preprocessing"""
        if not isinstance(text, str) or not text.strip():
            return ""

        # Unicode normalization for Devanagari script
        text = unicodedata.normalize('NFC', text)

        # Remove URLs, emails, and phone numbers
        text = re.sub(r'http[s]?://\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'\d{10,}', '', text)

        # Handle Marathi numerals
        text = re.sub(r'[०-९]', '', text)

        # Normalize whitespace and newlines
        text = re.sub(r'\s+', ' ', text)

        # Remove excessive punctuation
        text = re.sub(r'([।])\1+', r'\1', text)

        return text.strip()

    def tokenize_marathi(self, text: str) -> List[str]:
        """Advanced Marathi tokenization"""
        # Split on word boundaries while preserving Devanagari script
        tokens = re.findall(r'\w+', text, re.UNICODE)

        # Filter tokens to keep only Devanagari and meaningful length
        filtered_tokens = [
            token for token in tokens
            if len(token) > 1 and re.search(self.devanagari_range, token)
        ]

        return filtered_tokens

    def apply_light_stemming(self, word: str) -> str:
        """Apply light stemming for Marathi words"""
        # Remove common suffixes
        for suffix in self.marathi_suffixes:
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                return word[:-len(suffix)]

        # Remove common prefixes
        for prefix in self.marathi_prefixes:
            if word.startswith(prefix) and len(word) > len(prefix) + 2:
                return word[len(prefix):]

        return word

    @lru_cache(maxsize=1000)
    def preprocess_text(self, text: str, remove_stopwords: bool = True,
                       apply_stemming: bool = False) -> str:
        """
        Enhanced text preprocessing with caching

        Args:
            text: Input Marathi text
            remove_stopwords: Whether to remove stopwords
            apply_stemming: Whether to apply light stemming

        Returns:
            Preprocessed text
        """
        # Normalize text
        text = self.normalize_text(text)

        # Remove punctuation
        text = text.translate(str.maketrans('', '', self.marathi_punctuation))

        # Tokenize
        tokens = self.tokenize_marathi(text)

        if remove_stopwords:
            tokens = [token for token in tokens if token not in self.marathi_stopwords]

        if apply_stemming:
            tokens = [self.apply_light_stemming(token) for token in tokens]

        return ' '.join(tokens)

class EnhancedSimilarityCalculator:
    """Enhanced similarity calculation with multiple metrics"""

    def __init__(self, config: PlagiarismConfig):
        self.config = config
        self.text_processor = MarathiTextProcessor()

        # Initialize TF-IDF with optimized parameters
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=config.max_features,
            min_df=config.min_df,
            max_df=config.max_df,
            ngram_range=config.ngram_range,
            tokenizer=self.text_processor.tokenize_marathi,
            lowercase=True,
            strip_accents='unicode',
            token_pattern=r'\w+'
        )

        # Initialize BERT model with error handling
        try:
            # Updated to use available Marathi-compatible models
            available_models = [
                'l3cube-pune/marathi-bert',
                'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                'sentence-transformers/distiluse-base-multilingual-cased-v1'
            ]

            self.bert_model = None
            for model_name in available_models:
                try:
                    self.bert_model = SentenceTransformer(model_name)
                    logger.info(f"Loaded BERT model: {model_name}")
                    break
                except Exception as e:
                    logger.warning(f"Failed to load {model_name}: {e}")
                    continue

            if self.bert_model is None:
                raise Exception("All BERT models failed to load")

        except Exception as e:
            logger.error(f"Failed to load any BERT model: {e}")
            # Fallback to a simple embedding approach
            self.bert_model = None

        # Cache for embeddings
        self.embedding_cache = {}

    def calculate_tfidf_similarity_batch(self, queries: List[str],
                                       corpus_docs: List[str]) -> np.ndarray:
        """Calculate TF-IDF similarity in batches for better performance"""
        try:
            # Preprocess all documents
            processed_corpus = [
                self.text_processor.preprocess_text(doc) for doc in corpus_docs
            ]

            # Fit TF-IDF on corpus
            corpus_tfidf = self.tfidf_vectorizer.fit_transform(processed_corpus)

            # Process queries in batches
            similarities = []
            for i in range(0, len(queries), self.config.batch_size):
                batch_queries = queries[i:i + self.config.batch_size]
                processed_queries = [
                    self.text_processor.preprocess_text(q) for q in batch_queries
                ]

                query_tfidf = self.tfidf_vectorizer.transform(processed_queries)
                batch_similarities = cosine_similarity(query_tfidf, corpus_tfidf)
                similarities.extend(batch_similarities)

            return np.array(similarities)

        except Exception as e:
            logger.error(f"Error in TF-IDF similarity calculation: {e}")
            return np.zeros((len(queries), len(corpus_docs)))

    def calculate_bert_similarity_batch(self, queries: List[str],
                                      corpus_docs: List[str]) -> np.ndarray:
        """Calculate BERT similarity with optimization"""
        try:
            if self.bert_model is None:
                logger.warning("BERT model not available, returning zeros")
                return np.zeros((len(queries), len(corpus_docs)))

            # Check cache first
            corpus_key = hash(tuple(corpus_docs))
            if self.config.cache_embeddings and corpus_key in self.embedding_cache:
                corpus_embeddings = self.embedding_cache[corpus_key]
            else:
                # Encode corpus documents
                corpus_embeddings = self.bert_model.encode(
                    corpus_docs,
                    convert_to_tensor=True,
                    show_progress_bar=False,
                    batch_size=self.config.batch_size
                )
                if self.config.cache_embeddings:
                    self.embedding_cache[corpus_key] = corpus_embeddings

            # Encode queries
            query_embeddings = self.bert_model.encode(
                queries,
                convert_to_tensor=True,
                show_progress_bar=False,
                batch_size=self.config.batch_size
            )

            # Calculate similarities
            similarities = cosine_similarity(
                query_embeddings.cpu().numpy(),
                corpus_embeddings.cpu().numpy()
            )

            return similarities

        except Exception as e:
            logger.error(f"Error in BERT similarity calculation: {e}")
            return np.zeros((len(queries), len(corpus_docs)))

    def calculate_ensemble_similarity(self, tfidf_sim: np.ndarray,
                                    bert_sim: np.ndarray) -> np.ndarray:
        """Calculate weighted ensemble similarity"""
        return (self.config.tfidf_weight * tfidf_sim +
                self.config.bert_weight * bert_sim)

    def calculate_confidence_score(self, similarities: np.ndarray) -> np.ndarray:
        """Calculate confidence scores based on similarity distribution"""
        # Use statistical measures for confidence
        mean_sim = np.mean(similarities, axis=1, keepdims=True)
        std_sim = np.std(similarities, axis=1, keepdims=True)

        # Confidence based on how much higher the max similarity is
        max_sim = np.max(similarities, axis=1, keepdims=True)
        confidence = 1 - (max_sim - mean_sim) / (std_sim + 1e-8)

        return np.clip(confidence, 0, 1)

class EnhancedMarathiPlagiarismDetector:
    """Enhanced Marathi plagiarism detection with multiple improvements"""

    def __init__(self, config: Optional[PlagiarismConfig] = None):
        self.config = config or PlagiarismConfig()
        self.similarity_calculator = EnhancedSimilarityCalculator(self.config)
        self.corpus = []
        self.corpus_metadata = []

        # Performance metrics
        self.processing_times = []
        self.cache_hits = 0
        self.cache_misses = 0

    def build_corpus(self, documents: List[str],
                    document_names: Optional[List[str]] = None,
                    metadata: Optional[List[Dict[str, Any]]] = None):
        """Build enhanced corpus with metadata support"""
        if not documents:
            raise ValueError("Documents list cannot be empty")

        self.corpus = documents
        self.corpus_metadata = metadata or [{} for _ in documents]

        # Generate document names if not provided
        if document_names is None:
            document_names = [f"Document_{i+1}" for i in range(len(documents))]

        # Store document names in metadata
        for i, name in enumerate(document_names):
            self.corpus_metadata[i]['name'] = name
            self.corpus_metadata[i]['index'] = i
            self.corpus_metadata[i]['length'] = len(documents[i])
            self.corpus_metadata[i]['word_count'] = len(documents[i].split())

    def detect_plagiarism_single(self, query_text: str,
                                threshold: Optional[float] = None,
                                return_detailed: bool = True) -> Dict[str, Any]:
        """Enhanced single query plagiarism detection"""
        threshold = threshold or self.config.threshold

        if not self.corpus:
            return {"error": "Corpus not built. Please build corpus first."}

        if not query_text or not isinstance(query_text, str):
            return {"error": "Invalid query text"}

        try:
            start_time = pd.Timestamp.now()

            # Calculate similarities
            tfidf_sim = self.similarity_calculator.calculate_tfidf_similarity_batch(
                [query_text], self.corpus
            )[0]

            bert_sim = self.similarity_calculator.calculate_bert_similarity_batch(
                [query_text], self.corpus
            )[0]

            ensemble_sim = self.similarity_calculator.calculate_ensemble_similarity(
                tfidf_sim.reshape(1, -1), bert_sim.reshape(1, -1)
            )[0]

            confidence = self.similarity_calculator.calculate_confidence_score(
                ensemble_sim.reshape(1, -1)
            )[0]

            # Calculate processing time
            processing_time = (pd.Timestamp.now() - start_time).total_seconds()
            self.processing_times.append(processing_time)

            # Prepare results - FIXED: Use numpy.round() instead of round()
            max_similarity = float(np.max(ensemble_sim))
            max_index = int(np.argmax(ensemble_sim))

            results = {
                'query_text': query_text,
                'query_length': len(query_text),
                'query_word_count': len(query_text.split()),
                'threshold': threshold,
                'processing_time_seconds': round(processing_time, 3),
                'overall_analysis': {
                    'max_similarity': float(np.round(max_similarity * 100, 2)),
                    'max_similarity_document': self.corpus_metadata[max_index]['name'],
                    'is_plagiarized': max_similarity >= threshold,
                    'confidence_score': float(np.round(confidence * 100, 2)),
                    'suspected_sources': int(np.sum(ensemble_sim >= threshold)),
                    'similarity_percentile': float(np.round(
                        (np.sum(ensemble_sim < max_similarity) / len(ensemble_sim)) * 100, 1
                    ))
                },
                'method_breakdown': {
                    'tfidf_similarity': float(np.round(np.max(tfidf_sim) * 100, 2)),
                    'bert_similarity': float(np.round(np.max(bert_sim) * 100, 2)),
                    'ensemble_similarity': float(np.round(max_similarity * 100, 2))
                },
                'ensemble_weights': {
                    'tfidf_weight': self.config.tfidf_weight,
                    'bert_weight': self.config.bert_weight
                }
            }

            if return_detailed:
                detailed_results = []
                for i, (tfidf_score, bert_score, ensemble_score) in enumerate(
                    zip(tfidf_sim, bert_sim, ensemble_sim)
                ):
                    detailed_results.append({
                        'document_index': i,
                        'document_name': self.corpus_metadata[i]['name'],
                        'tfidf_similarity': float(np.round(tfidf_score * 100, 2)),
                        'bert_similarity': float(np.round(bert_score * 100, 2)),
                        'ensemble_similarity': float(np.round(ensemble_score * 100, 2)),
                        'is_plagiarized': ensemble_score >= threshold,
                        'document_preview': self.corpus[i][:100] + "..." if len(self.corpus[i]) > 100 else self.corpus[i]
                    })

                # Sort by ensemble similarity
                detailed_results.sort(key=lambda x: x['ensemble_similarity'], reverse=True)
                results['detailed_results'] = detailed_results
                results['top_matches'] = detailed_results[:5]

            return results

        except Exception as e:
            logger.error(f"Error in plagiarism detection: {e}")
            return {"error": f"Detection failed: {str(e)}"}

    def detect_plagiarism_batch(self, queries: List[str],
                               threshold: Optional[float] = None,
                               show_progress: bool = True) -> List[Dict[str, Any]]:
        """Batch plagiarism detection with progress tracking"""
        threshold = threshold or self.config.threshold

        if not self.corpus:
            return [{"error": "Corpus not built. Please build corpus first."}]

        results = []
        total_queries = len(queries)

        for i, query in enumerate(queries):
            if show_progress:
                print(f"Processing query {i+1}/{total_queries}...", end='\r')

            result = self.detect_plagiarism_single(query, threshold, return_detailed=False)
            results.append(result)

        if show_progress:
            print("\nBatch processing complete!")

        return results

    def get_performance_stats(self) -> Dict[str, Any]:
        """Get performance statistics"""
        if not self.processing_times:
            return {"error": "No processing data available"}

        return {
            'total_queries_processed': len(self.processing_times),
            'average_processing_time': round(float(np.mean(self.processing_times)), 3),
            'median_processing_time': round(float(np.median(self.processing_times)), 3),
            'min_processing_time': round(float(np.min(self.processing_times)), 3),
            'max_processing_time': round(float(np.max(self.processing_times)), 3),
            'cache_efficiency': {
                'hits': self.cache_hits,
                'misses': self.cache_misses,
                'hit_rate': round(float(self.cache_hits / (self.cache_hits + self.cache_misses + 1e-8)), 3)
            }
        }

    def optimize_weights(self, validation_data: List[Tuple[str, str, int]],
                        metric: str = 'f1') -> Dict[str, float]:
        """Optimize ensemble weights using validation data"""
        from sklearn.metrics import f1_score, precision_score, recall_score

        best_weights = {'tfidf_weight': self.config.tfidf_weight, 'bert_weight': self.config.bert_weight}
        best_score = 0

        weight_combinations = [(w/10, 1-w/10) for w in range(1, 10)]

        for tfidf_w, bert_w in weight_combinations:
            # Temporarily set weights
            original_tfidf = self.config.tfidf_weight
            original_bert = self.config.bert_weight

            self.config.tfidf_weight = tfidf_w
            self.config.bert_weight = bert_w

            # Calculate predictions
            predictions = []
            true_labels = []

            for query, corpus_doc, label in validation_data:
                self.build_corpus([corpus_doc], ['validation'])
                result = self.detect_plagiarism_single(query, return_detailed=False)

                if 'error' not in result:
                    pred = 1 if result['overall_analysis']['is_plagiarized'] else 0
                    predictions.append(pred)
                    true_labels.append(label)

            # Calculate metric
            if metric == 'f1' and len(set(true_labels)) > 1:
                score = f1_score(true_labels, predictions)
            elif metric == 'precision':
                score = precision_score(true_labels, predictions, zero_division=0)
            elif metric == 'recall':
                score = recall_score(true_labels, predictions, zero_division=0)
            else:
                score = 0

            # Restore original weights
            self.config.tfidf_weight = original_tfidf
            self.config.bert_weight = original_bert

            if score > best_score:
                best_score = score
                best_weights = {'tfidf_weight': tfidf_w, 'bert_weight': bert_w}

        return {
            'optimal_weights': best_weights,
            f'best_{metric}_score': round(best_score, 3),
            'optimization_complete': True
        }

class EnhancedMarathiDataset:
    """
    Simple dataset generator for Marathi plagiarism detection
    """

    def __init__(self):
        self.original_sentences = [
            "शेती हा भारतातील प्रमुख व्यवसाय आहे.",
            "मराठी भाषा ही भारतातील एक समृद्ध भाषा आहे.",
            "आरोग्य हे जीवनातील सर्वात मोठे संपत्ती आहे.",
            "शिक्षणामुळे माणसाचे जीवन बदलते.",
            "भारत हा विविध संस्कृतींचा देश आहे.",
            "तंत्रज्ञानामुळे जग खूप वेगाने बदलत आहे.",
            "विद्यार्थ्यांनी नियमित अभ्यास करणे आवश्यक आहे.",
            "व्यायाम केल्याने शरीर निरोगी राहते.",
            "महाराष्ट्राची राजधानी मुंबई आहे.",
            "संगणक तंत्रज्ञान आजच्या काळात खूप महत्त्वाचे आहे."
        ]

    def create_comprehensive_dataset(self, size=100):
        import random

        texts = []
        labels = []

        for _ in range(size):

            sentence = random.choice(self.original_sentences)

            # 50% original
            if random.random() < 0.5:
                texts.append(sentence)
                labels.append(0)

            # 50% plagiarized (slightly modified)
            else:
                plagiarized = sentence + " हे सर्वांना माहित आहे."
                texts.append(plagiarized)
                labels.append(1)

        df = pd.DataFrame({
            "text": texts,
            "label": labels
        })

        return df

# Example usage and demonstration
def main():
    """Main function demonstrating the enhanced plagiarism detection system"""

    print("Starting Enhanced Marathi Plagiarism Detection System...")

    # Initialize detector with custom configuration
    config = PlagiarismConfig(
        tfidf_weight=0.4,
        bert_weight=0.6,
        threshold=0.55,
        max_features=8000,
        min_df=1,
        max_df=0.95,
        ngram_range=(1, 3),
        batch_size=32,
        use_faiss=False,
        cache_embeddings=True,
        enable_parallel_processing=True
    )

    detector = EnhancedMarathiPlagiarismDetector(config)

    # Create comprehensive dataset
    logger.info("Creating comprehensive Marathi dataset...")
    dataset_creator = EnhancedMarathiDataset()
    dataset = dataset_creator.create_comprehensive_dataset(size=100)

    # Save dataset
    dataset.to_csv("marathi_plagiarism_dataset.csv", index=False, encoding='utf-8')
    logger.info(f"Dataset created with {len(dataset)} samples")

    # Split dataset for training and testing
    from sklearn.model_selection import train_test_split

    train_data, test_data = train_test_split(
        dataset, test_size=0.3, random_state=42, stratify=dataset['label']
    )

    # Build corpus from training data (original documents only)
    original_docs = train_data[train_data['label'] == 0]['text'].tolist()

    if len(original_docs) > 0:
        detector.build_corpus(
            original_docs,
            document_names=[f"original_{i}" for i in range(len(original_docs))]
        )
        logger.info(f"Corpus built with {len(original_docs)} original documents")
    else:
        # Fallback to using all training documents
        detector.build_corpus(
            train_data['text'].tolist(),
            document_names=[f"doc_{i}" for i in range(len(train_data))]
        )
        logger.info(f"Warning: No original documents found. Using all {len(train_data)} training documents")

    # Test single query
    logger.info("\nTesting single query...")
    test_query = "शेती हा भारतातील प्रमुख व्यवसाय आहे. पावसावर अवलंबून असलेल्या शेतीला सध्या अनेक समस्या आहेत."

    result = detector.detect_plagiarism_single(test_query, return_detailed=True)

    if 'error' not in result:
        print(f"\nQuery: {test_query[:100]}...")
        print(f"Status: {'Plagiarized' if result['overall_analysis']['is_plagiarized'] else 'Original'}")
        print(f"Max Similarity: {result['overall_analysis']['max_similarity']}%")
        print(f"Confidence: {result['overall_analysis']['confidence_score']}%")
        print(f"Processing Time: {result['processing_time_seconds']} seconds")
    else:
        print(f"Error: {result['error']}")

    print("\nEnhanced Marathi Plagiarism Detection System Demo Complete!")

if __name__ == "__main__":
    main()

Starting Enhanced Marathi Plagiarism Detection System...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: l3cube-pune/marathi-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Query: शेती हा भारतातील प्रमुख व्यवसाय आहे. पावसावर अवलंबून असलेल्या शेतीला सध्या अनेक समस्या आहेत....
Status: Plagiarized
Max Similarity: 91.75%
Confidence: 0.0%
Processing Time: 2.853 seconds

Enhanced Marathi Plagiarism Detection System Demo Complete!


In [ ]:
# -*- coding: utf-8 -*-
"""
Test Script for Marathi Plagiarism Detection System
Tests both plagiarized and original sentences
"""

import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def test_plagiarism_detection():
    """Test the plagiarism detection system with both plagiarized and original sentences"""

    print("="*70)
    print("MARATHI PLAGIARISM DETECTION - TEST DEMONSTRATION")
    print("="*70)

    # Initialize detector with optimized settings
    config = PlagiarismConfig(
        tfidf_weight=0.4,
        bert_weight=0.6,
        threshold=0.55,  # 55% similarity threshold
        max_features=8000,
        min_df=1,
        max_df=0.95,
        ngram_range=(1, 3),
        batch_size=16,
        cache_embeddings=True
    )

    detector = EnhancedMarathiPlagiarismDetector(config)

    # Create a small test corpus with original documents
    print("\n1. Creating test corpus with original Marathi documents...")

    # Original documents (reference corpus)
    original_documents = [
        """जॉर्जला नीओशोत पोहोचायला रात्र झाली. एका पडक्या गोठ्यात त्याने आपलं चालून चालून थकलेलं दुबळं शरीर आडवं केलं. पहाटेच उठला. आपल्या शेजारी कोण म्हणून पाहतो तो एक भला दांडगा कुत्रा ! बापरे !! रात्र याच्या संगतीत गेली तर !!
त्या सकाळी आयुष्यात प्रथमच त्याने शाळेत पाऊल टाकलं. सारी निग्रोच मुलं होती. जॉर्जचं ते गबाळं ध्यान, डोक्यावर गोठ्यातल्या गवताची चिकटलेली तुसं आणि त्याचा तो चिरका आवाज ! अडखळत बोलणं! सारी मुलं त्याच्याकडे विचित्र नजरेने पाहत होती.
शाळा सुटल्यावर मुलं पांगली. कुठे जावं हे न समजल्याने जॉर्ज वर्गातच बसून राहिला. शेवटी त्याच्या शिक्षकांनीच त्याला त्याच्या घरी जायला सांगितलं. तो बाहेर पडला. सूझनबाईंच्या आदेशाप्रमाणे घरोघर काम मागत हिंडला. पण एवढ्याशा पोराची मदत कोणाला हवी असणार?
नंतरचे काही दिवस पडतील ती, मिळतील ती कामं तो हरकाम्यासारखी करत राहिला. मिळवलेल्या दिडक्यांतून दुकानात जाऊन खाणं विकत घेता येतं हे शिकण्यासाठी त्याला दोन दिवस उपाशीपोटी राहावं लागलं. एकदा तर एका दुकानदाराने उचल्या दिसतोय म्हणून थप्पडही लगावली. हाही एक नवा धडा !
दिवसभर शाळा आणि त्यानंतर पोटाचा प्रश्न मिटवण्यासाठी मरमर काबाडकष्ट केल्यावर रात्री झोपायला जुना पडका गोठा होताच. पुन्हा पहाटे उठून उगवणारा दिवस पाठी टाकण्यासाठी धडपड सुरू. पण शाळा मात्र एकही दिवस चुकवली नाही. त्याच्या बोलण्यात आता खूप सुधारणा झाली होती. अडखळणंही बरंच कमी झालं होतं. आवाज मात्र होता तसाच वरच्या पट्टीत राहिला होता, तरी चिरकेपणा बराच कमी झाला होता.
नीओशोमधला तो पहिलाच हिवाळा जॉर्जच्या आठवणीत जन्मभर राहिला. रक्त गोठवणारी ती मरणाची थंडी त्याला रोज सकाळी आपण गेली रात्र कशी तग धरली असा प्रश्न टाकी. तो त्या भयानक गारठ्यात अन्नाशिवाय, उबदार कपड्यांशिवाय कसा जगला हे एक गूढच आहे. पण तो जगला. त्या गोठवणाऱ्या थंडीचे परिणाम जन्मभर सोशीत तो जगला हे मात्र नक्की!!
वसंत ऋतू आला. त्या काळी वसंत ऋतू सुरू होताच निग्रो मुलांच्या शाळांना सुट्टी मिळत असे. शेतात लावणी-पेरणीच्या कामी मुलांचा पालकांना उपयोग व्हावा हा उद्देश! आपण या सुट्टीत काय करावं याची योजना जॉर्जने आखली होती. नीओशोच्या गावकुसाबाहेर दूर राहणाऱ्या द्राक्षवाल्या यायगरसाहेबांची भेट घ्यायची होती. त्यांनी दिलेलं पुस्तक वाचता यावं म्हणून तो शिक्षण घेण्याची धडपड करतोय, हे त्यांना सांगायचं होतं. उपासमार सहन केली
पण शिक्षण सोडलं नाही हे सांगून पाठ थोपटून घ्यायची होती. यायगरसाहेबांच्या भेटीची स्वप्नं रंगवत मोठ्या उत्साहाने तो चांगला कोस-दोन कोस चालून यायगरच्या मळ्याशी पोहोचला. फाटकापाशीच समजलं, यायगर स्वर्गवासी होऊन दोन महिने लोटले. तो सुन्न झाला. ज्या भयानक थंडीतून जॉर्ज कसाबसा वाचला होता, त्याच थंडीने यायगरचा बळी घेतला होता. तो उदास,
खिन्न मनाने परतला. पुढील आयुष्यात त्या प्रसंगाची आठवण देताना जॉर्ज म्हणे, "मी त्या बागेपासून परतलो आणि पुन्हा गावात कसा आलो, हे मला त्या वेळी कळलं नाहीच आणि पुढेही कधी ते कोडं उलगडलं नाही. किती निराशा आणि विमनस्कता अनुभवली त्या वेळी मी!"
नेहमीप्रमाणे त्या जुन्या-पडक्या गोठ्यात झोपला असताना कोणाच्या तरी ढोसण्याने जॉर्जला जाग आली. त्याला वाटलं, आता आपली घटका भरली. गोठ्याच्या मालकाला आपला सुगावा लागलेला दिसतो. आता आपली दोन-चार तरी हाडं मोडणार या भीतीने जॉर्ज अर्धमेला झाला. थकलेला, सदाभुकेला जॉर्ज गळा काढून रडू लागला.
पण घडलं वेगळंच ! तो ढोसणारा बुवा त्याला हलके हलके थोपटत म्हणाला, "उगी, उगी. रडू नकोस. मी काही तुला मारण्यासाठी नाही उठवलं. कोण आहेस रे बाळा तू? इथे या पडक्या गोठ्यात का झोपला आहेस?"आधीच बोलणं अडखळत आणि त्यात हुंदक्यांची भर. जॉर्जने कशीबशी आपली कर्मकहाणी त्या बुवाला सांगितली.
पण त्या भल्या माणसाने जॉर्जला धीर दिला. म्हणाला,"बाळा, घाबरू नकोस. माझ्याबरोबर चल."त्या ढोसणाऱ्या बुवाचं नाव जॉन मार्टिन. जॉर्जला घेऊन जॉन आपल्या घरी आला. त्याच्या घरातल्या स्वच्छ टेबलावर, कित्येक महिन्यांच्या अवधीनंतर, ज्याला जेवण म्हणता येईल असं अन्न जॉर्ज प्रथमच जेवला,
जेवल्यानंतर त्याला दिलेल्या बाजल्यावर तो चटकन झोपी गेला. भरल्यापोटी कशी मस्त झोप लागली.जॉर्ज आता मार्टिनकुटुंबातच राहू लागला. जॉन मार्टिन एका पिठाच्या गिरणीवर कामाला होता. मार्टिन दांपत्याची आर्थिक स्थिती जेमतेमच होती. पण गरीब निग्रोंविषयी, गुलामांविषयी आस्था असल्याने जॉन आपल्या परीने त्यांना मदत करत असे.
मूलबाळ नसलेल्या मार्टिन दांपत्याने जॉर्जला आपला मुलगाच मानला. त्यांना कधी वाटलंही नसेल, की या दिवसाचे त्यांना आभार मानावे लागणार आहेत. या कृत्याबद्दल 'धन्य, धन्य' वाटणार आहे.दुसऱ्या दिवसापासूनच जॉर्जचे छुपे गुण, त्याचं पाक-कौशल्य, स्वच्छता, टापटीप मार्टिनच्या ध्यानात आली. त्यांच्या दारची बाग तर कधी नव्हती इतकी स्वच्छ झाडलेली होती. झाडांच्या आळ्यांनाही कसा छान आकार आला होता.
शिवाय आता आणखी नवीन बियाण्यांचीही मागणी जॉर्जने केली होती.मार्टिन दांपत्याला जॉर्जच्या कष्टाचा मोबदला देणं आर्थिकदृष्ट्या शक्य नव्हतं. म्हणून त्यांनी त्याला बाहेरची कामं शोधण्याचा सल्ला दिला. सौ. मार्टिनच्या आधीच उठून, घरकाम आटपून, मग न्याहारी उरकून कामाच्या शोधात जॉर्ज गावं पालथा घाली. पोटापाण्याची चिंता मिटल्याने तो
आता नव्या उमेदीने वावरू लागला. लोकही त्याला त्याच्या गुणांमुळे, कामसू वृत्तीमुळे जवळ करू लागले होते. तोही आता हळूहळू आपला बुजरेपणा सोडून माणसांत मिसळू लागला होता. जॉन मार्टिनला जॉर्जच्या अभ्यासू वृत्तीचा, उत्सुक नजरेचा, तल्लख बुद्धीचा अंदाज येऊ लागला. सर्व बाबतींत जॉन जॉर्जला मदत करत होता, प्रोत्साहन देत होता.
जॉर्ज अजूनही पुरत्या मोकळेपणाने, खुल्या वृत्तीने वागत नाही हे मार्टिनच्या ध्यानी आल्यावर त्याने त्यावर विचार केला. कदाचित त्याला आपल्या 'स्वतःच्या' लोकांत मिसळणं आवडेल, असा हिशेब करून त्या उद्देशाने त्याने गावातील निग्रोंसाठी असलेल्या चर्चमध्ये जॉर्जला पाठवण्याचं ठरवलं.
जॉन मार्टिनच्या सूचनेप्रमाणे जॉर्ज त्या रविवारी चर्चला गेला. सारेजण कसे सजून आले होते! जॉर्ज अगदी पाठीमागच्या कोपऱ्यात अंग चोरून बसला.प्रार्थनेला सुरुवात झाली. पदं गायला सुरुवात झाली. त्या सुरांनी जॉर्जचं भान हरपलं. त्याची आई असेच सूर गुणगुणून त्याला जोजवत होती का? त्याच्या भावनेची तार झंकारली. भान हरपून तो ऐकत राहिला. नंतरचं प्रवचन मात्र त्याला अजिबात आवडलं नाही. त्याच्या मनाला विचार शिवून गेला-यायगरला समजला इतका अन्य कुणाला देव समजला असेल?
त्याच चर्चमध्ये मारियाआत्या नावाने ओळखल्या जाणाऱ्या एका भल्याभक्कम बाईशी त्याची ओळख झाली. शरीराप्रमाणे मनही मोठं होतं त्या आत्याबाईचं ! तिने मोठ्या कनवाळूपणे जॉर्जला थोपटलं. त्याची चौकशी केली. जॉर्जच्या मनाशी मारियाआत्याने एक वेगळीच जवळीक साधली. तिच्या स्वच्छ कपड्यांच्या सळसळीने, मायेच्या शब्दांनी त्याला सूझनबाईंची आठवण झाली.
लवकरच जॉर्जला चर्चचा कंटाळा आला. चर्चच्या वेळी बाहेरच्या हिरवळीवर पडून पदांच्या सुरावटी ऐकणं त्याला अधिक आवडे. पूढे पुढे तर त्याला हा 'उघड्यावरचा' देवाचा सहवास अधिकाधिक आवडू लागला. निसर्गदेवतेच्या सान्निध्यात रमणं हा तर पुढे त्याचा स्थायिभावच

"""
    ]

    # FIX: document_names must match number of documents
    detector.build_corpus(
        original_documents,
        document_names=["Marathi_Language_Doc"]
    )

    print(f"✓ Corpus built with {len(original_documents)} documents")

    # Test Cases
    print("\n2. Preparing test sentences...")

    # Test Case 1: PLAGIARIZED sentence
    plagiarized_sentence = """जॉर्जला नीओशोत पोहोचायला रात्र झाली. एका पडक्या गोठ्यात त्याने आपलं चालून चालून थकलेलं दुबळं शरीर आडवं केलं.
    पहाटेच उठला. आपल्या शेजारी कोण म्हणून पाहतो तो एक भला दांडगा कुत्रा ! बापरे !! रात्र याच्या संगतीत गेली तर !!
त्या सकाळी आयुष्यात प्रथमच त्याने शाळेत पाऊल टाकलं."""

    # Test Case 2: ORIGINAL sentence
    original_sentence = """शेती हा भारतातील प्रमुख व्यवसाय आहे. पावसावर अवलंबून असलेल्या शेतीला
        सध्या अनेक समस्या आहेत. शेतकऱ्यांना पिकांच्या योग्य किमत मिळत नाहीत
        आणि त्यामुळे त्यांचे आर्थिक स्थिती बिघडते."""

    # Test Case 3: PARAPHRASED sentence
    paraphrased_sentence = """जॉर्ज त्या काळात फक्त तेरा वर्षांचा असला तरी तो कपडे धुण्याचं काम खूप मन लावून करत होता.
धुलाईपासून इस्त्री करून नीट घड्या घालण्यापर्यंत प्रत्येक गोष्ट तो काळजीपूर्वक आणि नियोजनाने पार पाडत असे.
फोर्ट स्कूलमध्ये प्रवेश घेण्यापूर्वीच त्याची ओळख अनेक शिक्षकांशी झाली होती.
शाळेत गेल्यानंतर त्याला लिहिण्यात आणि गणितात अडचणी येत होत्या, परंतु सृष्टिविज्ञानाबद्दलचं त्याचं ज्ञान शिक्षकांनाही आश्चर्यचकित करणारं होतं.
तो नेहमी काहीतरी वाचत असे आणि मोकळ्या वेळेत कागदावर निसर्गातील विविध गोष्टींची चित्रं रेखाटत असे."""

    test_sentences = [
        ("PLAGIARIZED (Direct Copy)", plagiarized_sentence),
        ("ORIGINAL (New Content)", original_sentence),
        ("PARAPHRASED (Similar Meaning)", paraphrased_sentence)
    ]

    print("✓ Test sentences prepared")

    # Test each sentence
    print("\n3. Testing plagiarism detection...")
    print("-" * 55)

    for test_type, sentence in test_sentences:
        print(f"\n📝 TESTING: {test_type}")
        print(f"Text: {sentence[:1200]}...")

        # Detect plagiarism
        result = detector.detect_plagiarism_single(sentence, return_detailed=True)

        if 'error' not in result:

            is_plagiarized = result['overall_analysis']['is_plagiarized']
            similarity = result['overall_analysis']['max_similarity']
            confidence = result['overall_analysis']['confidence_score']
            top_match = result['top_matches'][0] if result['top_matches'] else None

            print(f"📊 RESULTS:")
            print(f"  Status: {'🔴 PLAGIARIZED' if is_plagiarized else '🟢 ORIGINAL'}")
            print(f"  Max Similarity: {similarity}%")
            print(f"  Confidence: {confidence}%")
            print(f"  Processing Time: {result['processing_time_seconds']}s")

            if top_match:
                print(f"  Top Match: {top_match['document_name']} ({top_match['ensemble_similarity']}%)")

            print(f"  Method Breakdown:")
            print(f"    TF-IDF: {result['method_breakdown']['tfidf_similarity']}%")
            print(f"    BERT: {result['method_breakdown']['bert_similarity']}%")
            print(f"    Ensemble: {result['method_breakdown']['ensemble_similarity']}%")

            if test_type == "PLAGIARIZED (Direct Copy)" and is_plagiarized:
                print("  ✅ CORRECT: System detected plagiarism")
            elif test_type == "ORIGINAL (New Content)" and not is_plagiarized:
                print("  ✅ CORRECT: System detected original content")
            elif test_type == "PARAPHRASED (Similar Meaning)":
                print(f"  ℹ️  PARAPHRASED: Similarity {similarity}% (threshold: 55%)")
            else:
                print(f"  ⚠️  Review needed: Expected different result")

        else:
            print(f"❌ ERROR: {result['error']}")

        print("-" * 55)

    # Performance summary
    print("\n4. Performance Summary:")
    stats = detector.get_performance_stats()

    if 'error' not in stats:
        print(f"Total queries processed: {stats['total_queries_processed']}")
        print(f"Average processing time: {stats['average_processing_time']}s")
        print(f"Cache hit rate: {stats['cache_efficiency']['hit_rate']:.3f}")

    print("\n" + "="*55)
    print("TEST COMPLETED")
    print("="*55)


if __name__ == "__main__":
    test_plagiarism_detection()

MARATHI PLAGIARISM DETECTION - TEST DEMONSTRATION


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: l3cube-pune/marathi-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
ERROR:__main__:Error in TF-IDF similarity calculation: max_df corresponds to < documents than min_df



1. Creating test corpus with original Marathi documents...
✓ Corpus built with 1 documents

2. Preparing test sentences...
✓ Test sentences prepared

3. Testing plagiarism detection...
-------------------------------------------------------

📝 TESTING: PLAGIARIZED (Direct Copy)
Text: जॉर्जला नीओशोत पोहोचायला रात्र झाली. एका पडक्या गोठ्यात त्याने आपलं चालून चालून थकलेलं दुबळं शरीर आडवं केलं. 
    पहाटेच उठला. आपल्या शेजारी कोण म्हणून पाहतो तो एक भला दांडगा कुत्रा ! बापरे !! रात्र याच्या संगतीत गेली तर !!
त्या सकाळी आयुष्यात प्रथमच त्याने शाळेत पाऊल टाकलं....


ERROR:__main__:Error in TF-IDF similarity calculation: max_df corresponds to < documents than min_df


📊 RESULTS:
  Status: 🔴 PLAGIARIZED
  Max Similarity: 55.88%
  Confidence: 100.0%
  Processing Time: 2.202s
  Top Match: Marathi_Language_Doc (55.88%)
  Method Breakdown:
    TF-IDF: 0.0%
    BERT: 93.12999725341797%
    Ensemble: 55.88%
  ✅ CORRECT: System detected plagiarism
-------------------------------------------------------

📝 TESTING: ORIGINAL (New Content)
Text: शेती हा भारतातील प्रमुख व्यवसाय आहे. पावसावर अवलंबून असलेल्या शेतीला
        सध्या अनेक समस्या आहेत. शेतकऱ्यांना पिकांच्या योग्य किमत मिळत नाहीत
        आणि त्यामुळे त्यांचे आर्थिक स्थिती बिघडते....


ERROR:__main__:Error in TF-IDF similarity calculation: max_df corresponds to < documents than min_df


📊 RESULTS:
  Status: 🟢 ORIGINAL
  Max Similarity: 44.9%
  Confidence: 100.0%
  Processing Time: 0.472s
  Top Match: Marathi_Language_Doc (44.9%)
  Method Breakdown:
    TF-IDF: 0.0%
    BERT: 74.83999633789062%
    Ensemble: 44.9%
  ✅ CORRECT: System detected original content
-------------------------------------------------------

📝 TESTING: PARAPHRASED (Similar Meaning)
Text: जॉर्ज त्या काळात फक्त तेरा वर्षांचा असला तरी तो कपडे धुण्याचं काम खूप मन लावून करत होता. 
धुलाईपासून इस्त्री करून नीट घड्या घालण्यापर्यंत प्रत्येक गोष्ट तो काळजीपूर्वक आणि नियोजनाने पार पाडत असे. 
फोर्ट स्कूलमध्ये प्रवेश घेण्यापूर्वीच त्याची ओळख अनेक शिक्षकांशी झाली होती. 
शाळेत गेल्यानंतर त्याला लिहिण्यात आणि गणितात अडचणी येत होत्या, परंतु सृष्टिविज्ञानाबद्दलचं त्याचं ज्ञान शिक्षकांनाही आश्चर्यचकित करणारं होतं. 
तो नेहमी काहीतरी वाचत असे आणि मोकळ्या वेळेत कागदावर निसर्गातील विविध गोष्टींची चित्रं रेखाटत असे....
📊 RESULTS:
  Status: 🟢 ORIGINAL
  Max Similarity: 50.76%
  Confidence: 100.0%
  Processing Time: 1.05